> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG9KgDFL-g/JuY78K8O2tquhxpDmVhlSA/view?utm_content=DAG9KgDFL-g&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h89f16422ac)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install "langgraph-cli[inmem]" openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. Handoffs 模式

## 2.1 简介
假如使用 Handoffs 的方式构建多智能体系统，通常分为以下几步：
- 分析任务内容并拆解成多个连续且独立的子项目
- 定义需要保存的内部状态信息（AgentState）
- 创建管理内部状态信息的工具
- 定义每个阶段的系统提示词、工具和前置条件
- 创建按阶段动态切换配置的中间件
- 将记忆、工具、模型、内部状态信息和中间件进行组合形成智能体并进行调用

下面就让我们来开始真正的任务实战吧！

## 2.2 任务简介

在该案例中，我们要做一个客服智能体，该智能体主要完成的任务包括：
- 先收集产品的保修信息（不确认保修，后续能力不解锁）
- 再把问题分类成硬件/软件
- 最后根据保修情况以及软硬件信息给出对应的解决方案，或者升级到人工

根据刚刚任务的介绍，我们可以知道整个流程大体分成三个阶段：
- warranty_collector：收集保修
- issue_classifier：分类软硬件
- resolution_specialist：输出解决方案/转人工
所以根据这三阶段，我们可以先用 Literal 将几个可选项进行设置：

In [ ]:
from typing import Literal

# 可能的阶段名（workflow steps）
SupportStep = Literal[
    "warranty_collector",
    "issue_classifier",
    "resolution_specialist",
]

## 2.3 定义状态信息

接下来就是定义在智能体运行过程中需要保存的状态信息，包括：
- current_step：当前阶段信息，可选项为前面定义的阶段信息
- warranty_status：保修状态信息，分为在保修期和不在保修期
- issue_type：问题的具体类型，包括硬件和软件

In [ ]:
from langchain.agents import AgentState 
from typing_extensions import NotRequired 
from typing import Literal

class SupportState(AgentState):
    """客服流程的 state。"""
    current_step: NotRequired[SupportStep]  # 当前处于哪个阶段（核心字段）
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]  # 保修状态
    issue_type: NotRequired[Literal["hardware", "software"]]  # 问题类型

## 2.4 创建管理内部状态信息的工具
在设置好了每一步的流程和内部状态切换后，这里我们就可以开始设置工具让智能体拥有切换状态的能力了。

前面前期准备的时候也是提到了切换的话通常是通过 Command 方法来进行更新实现的，这里我们需要准备三个不同的工具来针对不同的阶段：
- 记录保修状态工具（record_warranty_status）
- 记录问题类型工具（record_issue_type）
- 解决/转人工工具（provide_solution / escalate_to_human）

### 2.4.1 记录保修状态工具

对于记录保修状态而言，并不说在工具里让大模型去分析是不是应该跳转，而是智能体分析完后认为要跳转的时候去调用这个工具实现跳转。

所以我们只需要完成一个最简单的任务，那就是获取到保修状态信息（status）后直接用 Command 进行跳转即可。

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from typing import Literal

@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """记录客户保修状态，并切换到【问题分类】阶段。"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"保修状态已记录：{status}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",
        }
    )

### 2.4.2 记录问题类型工具
与前面的工具类似，当智能体认为在与客户的对话中获取到了软硬件问题的信息后，才会调用该工具。

因此这里面我们还是更新三部分的信息：
- messages：返回的工具信息
- issue_type：记录的软/硬件问题
- current_step：准备进入下一阶段的步骤名称


In [ ]:
@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """记录问题类型，并切换到【解决方案】阶段。"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"问题类型已记录：{issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",
        }
    )

### 2.4.3 解决/转人工工具
最后到了解决方案的阶段，这里我们只需要根据前面的信息回复用户即可，因此这里不再需要更新内部的状态信息了，直接回复即可。

但这里只是模拟了可能交给人来解决的情况以及是直接提供解决方案的情况。在真实项目可替换为工单系统、通知系统等。

In [ ]:
@tool
def escalate_to_human(reason: str) -> str:
    """升级到人工支持。"""
    return f"已为你升级到人工支持。原因：{reason}"


@tool
def provide_solution(solution: str) -> str:
    """向客户提供解决方案。"""
    return f"已提供解决方案：{solution}"

## 2.5 提示词设置
在配置完工具以后，我们就可以来设置不同阶段下的提示词和工具了。

### 2.5.1 确定保修期
对于第一个阶段（确定保修期），我们需要的是让其友好的了解用户的产品是否在保修期内，所以提示词的设计如下：

In [ ]:
WARRANTY_COLLECTOR_PROMPT = """你是一名售后客服助手，正在帮助用户解决设备问题。

【当前阶段：确认保修】
你需要：
1. 友好地问候用户
2. 询问设备是否仍在保修期（或是否能提供购买时间/订单信息来判断）
3. 一旦信息足够明确，必须调用 record_warranty_status 记录结果并进入下一阶段

要求：语气自然、友好，不要一次问太多问题。"""

### 2.5.2 软硬件问题

然后在获取保修期信息后，第二阶段是了解是硬件还是软件的问题，这个时候就要让智能体能够友善的去进行询问，并且也给出一些判断的规则，只有明确了以后才去调用 record_issue_type 工具：

In [ ]:
ISSUE_CLASSIFIER_PROMPT = """你是一名售后客服助手，正在帮助用户解决设备问题。

【当前阶段：问题分类】
已知信息：保修状态 = {warranty_status}

你需要：
1. 引导用户描述问题现象
2. 判断问题属于【硬件】还是【软件】
   - 硬件：物理损坏/按键失灵/屏幕碎裂/进水等
   - 软件：系统卡顿/应用闪退/无法登录/网络配置等
3. 一旦判断足够明确，必须调用 record_issue_type 记录分类并进入下一阶段
如果不明确，可以继续追问，但不要武断下结论。"""

### 2.5.3 给出对应的解决方案

最后就是基于前面的内容给出对应的解决方案，软件类的问题就直接给出排查步骤并且进行解决。硬件问题就要看是否在保修期内，假如在的话就给出流程来回复，假如不在的话就人工介入来给出付费选择：

In [ ]:
RESOLUTION_SPECIALIST_PROMPT = """你是一名专业的售后支持工程师，正在帮助用户解决设备问题。

【当前阶段：给出解决方案】
已知信息：
- 保修状态 = {warranty_status}
- 问题类型 = {issue_type}
你需要：
1. 如果是【软件问题】：给出清晰的排查/修复步骤（从低风险到高风险），然后调用 provide_solution 工具进行回复
2. 如果是【硬件问题】：
   - 在保修期：说明官方保修维修流程、备份与注意事项，同时调用 provide_solution 工具进行回复
   - 不在保修期：调用 escalate_to_human，把用户转给人工说明付费维修选择
要求：回复具体、可执行、条理清晰。"""

## 2.5.4 STEP_CONFIG 字典创建
设置完提示词后，我们就可以把每个阶段所需要的提示词、所能使用的工具以及对应的前置条件组合到一起形成一个 config 文件供后续模型调用时进行使用：

In [ ]:
STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}

## 2.6 中间件创建

基于前面的内容，接下来我们就可以创建最核心的判断逻辑了，由于这里是在大模型调用时切换其能够使用的工具和中间件，因此这里所使用的就是 wrap_model_call 来进行实现：

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def apply_step_config(request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    # 1) 读取当前阶段（第一轮默认从保修确认开始）
    current_step = request.state.get("current_step", "warranty_collector")
    # 2) 获取该阶段配置
    stage_config = STEP_CONFIG[current_step]
    # 3) 校验必需字段
    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"在进入 {current_step} 之前，必须先设置 {key}")
    # 4) 格式化 prompt 并覆盖 system prompt + tools
    system_prompt = stage_config["prompt"].format(**request.state)

    request = request.override(system_prompt=system_prompt,
        tools=stage_config["tools"])
    # 5) 调用模型
    return handler(request)

## 2.7 创建智能体

接下来我们就可以开始组合对应的智能体信息了，这里我们需要五部分的内容：
- model：模型信息，也就是最开始定义的 ChatTongyi

In [ ]:
from langchain_community.chat_models import ChatTongyi
import os
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

- tools：工具信息，这里就是我们刚刚创建四个工具的集合列表

In [ ]:
all_tools = [record_warranty_status, record_issue_type, provide_solution, escalate_to_human]

- state_schema：这个就是内部状态的 AgentState 信息，也就是最开始定义的 SupportState
- middleware：中间件信息，也就是刚刚设置的 apply_step_config
- checkpointer：记忆信息，这里使用的就是 InMemorySaver() 来保存下短期的记忆，对于多智能体来说，由于 handoffs 的状态（current_step 等）要跨轮保存，假如没有 checkpointer，每轮都会像“第一轮”，所以是一定要进行设置的。

最终将所有内容组合在一起后可得到：

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)

## 2.8 调用智能体
创建好智能体后我们就可以对智能体进行调用了，但是首先我们要设置一个 thread_id 作为记忆的保存标记，这样就会把多轮对话的记忆进行保存：

In [ ]:
import uuid

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

然后我们就可以进行调用了：

In [ ]:
from langchain.messages import HumanMessage
# Turn 1: 用户报问题（进入保修确认）
r1 = agent.invoke({"messages": [HumanMessage("你好，我的手机屏幕摔裂了")]}, config)

# Turn 2: 用户回答保修（工具写入 state，切到分类）
r2 = agent.invoke({"messages": [HumanMessage("还在保修期，去年买的")]}, config)

# Turn 3: 用户描述现象（分类硬件/软件）
r3 = agent.invoke({"messages": [HumanMessage("就是屏幕有明显裂纹，触摸也不太灵敏")]}, config)

# Turn 4: 进入解决方案阶段
r4 = agent.invoke({"messages": [HumanMessage("那我现在应该怎么处理？")]}, config)

for msg in r4["messages"]:
    msg.pretty_print()

# 3. 进阶功能

## 3.1 添加记忆管理工具

前面我们基于内部状态机制、工具调用以及中间件成功实现了 Handoffs 系统的实现，但是由于这里我们并未真正的使用多智能体来对上下文进行管理，所以多轮对话过后我们会发现 token 消耗量越来越大。

因此可以给当前的智能体加上一些记忆管理的工具，比如内置中间件中的 SummarizationMiddleware 来实现记忆的总结来压缩上下文信息。

这里总结的逻辑是在总 token 数量大于 4000 时触发，并且至少保留 10 条上下文信息，其他的就通过 summarizer 进行总结。

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

summarizer = ChatTongyi(
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    model="qwen-plus",
    temperature=0.2,
)

agent = create_agent(
    model=model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[
        apply_step_config,
        SummarizationMiddleware(
            model=summarizer,
            trigger=("tokens", 4000),
            keep=("messages", 10),
        ),
    ],
    checkpointer=InMemorySaver(),
)

## 3.2 增加“回退能力”
那在真实的客服场景中，用户可能会改口说别的内容，比如：
- “啊不对，我其实不在保修期”
- “我想起来不是软件，是屏幕碎了”

这个时候我们假如一直按流程走的话智能体可能就懵逼了。因此为了解决这个问题我们可以添加两个回退工具，从而使其在遇到类似的场景时能够退到合适的场合：


In [ ]:
@tool
def go_back_to_warranty(runtime: ToolRuntime[None, SupportState]) -> Command:
    """回到保修确认阶段。"""
    return Command(
        update={"messages": [
                ToolMessage(
                    content="已返回保修确认阶段",
                    tool_call_id=runtime.tool_call_id)],
            "current_step": "warranty_collector"})

@tool
def go_back_to_classification(
    runtime: ToolRuntime[None, SupportState]) -> Command:
    """回到问题分类阶段。"""
    return Command(update={
            "messages": [
                ToolMessage(
                    content="已返回问题分类阶段",
                    tool_call_id=runtime.tool_call_id)],
            "current_step": "issue_classifier"})

增加工具后我们也要同步更新 STEP_CONFIG 内容：

In [ ]:
STEP_CONFIG = {
  "warranty_collector": {
    "prompt": WARRANTY_COLLECTOR_PROMPT,
    "tools": [record_warranty_status],
    "requires": [],
  },
  "issue_classifier": {
    "prompt": ISSUE_CLASSIFIER_PROMPT,
    "tools": [record_issue_type, go_back_to_warranty],
    "requires": ["warranty_status"],
  },
  "resolution_specialist": {
    "prompt": RESOLUTION_SPECIALIST_PROMPT,
    "tools": [provide_solution, escalate_to_human, go_back_to_warranty, go_back_to_classification],
    "requires": ["warranty_status", "issue_type"],
  },
}

当然在所有的工具箱上也要同步将这两个工具给加载进去：

In [ ]:
all_tools = [record_warranty_status, record_issue_type, provide_solution, 
       escalate_to_human, go_back_to_warranty, go_back_to_classification]

除此之外，RESOLUTION_SPECIALIST 阶段和 ISSUE_CLASSIFIER 阶段的提示词也需要进行更新：

In [ ]:
ISSUE_CLASSIFIER_PROMPT = """你是一名售后客服助手，正在帮助用户解决设备问题。
【当前阶段：问题分类】
已知信息：保修状态 = {warranty_status}
你需要：
1. 引导用户描述问题现象
2. 判断问题属于【硬件】还是【软件】
3. 一旦判断足够明确，必须调用 record_issue_type 记录分类并进入下一阶段

如果用户纠正了信息：
- 用 go_back_to_warranty 回到保修确认

如果不明确，可以继续追问，但不要武断下结论。"""

RESOLUTION_SPECIALIST_PROMPT = """你是一名专业的售后支持工程师，正在帮助用户解决设备问题。
【当前阶段：给出解决方案】
已知信息：
- 保修状态 = {warranty_status}
- 问题类型 = {issue_type}
你需要：
1. 如果是【软件问题】：调用 provide_solution 给出清晰的排查/修复步骤（从低风险到高风险）
2. 如果是【硬件问题】：
  - 在保修期：调用 provide_solution 说明官方保修维修流程、备份与注意事项
  - 不在保修期：调用 escalate_to_human 转人工说明付费维修选择
如果用户纠正了信息：
- 用 go_back_to_warranty 回到保修确认
- 用 go_back_to_classification 回到问题分类
要求：回复具体、可执行、条理清晰。"""

更新完后我们再对 agent 进行同样的组装后，就可以进行调用测试了：

In [ ]:
agent = create_agent(
  model=model,
  tools=all_tools,
  state_schema=SupportState,
  middleware=[apply_step_config],
  checkpointer=InMemorySaver(),
)